In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from configs.config import DATA_DIR, FEWSHOT_CACHE_DIR, PROMPT_DIR, GREEDY_CONFIG
import json
from src.extractor import LabelTransformConfig, prepare_label_tokens, _parse_parent_annotations
    
from src.tokenizer_utils import tokenize, decode
from src.htmlLabel import simplified_to_normal_form
from src.models import get_messages
from tqdm import tqdm

c:\Users\zakga\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Choose your file

In [2]:
filename = "2021QCCA1675"
split = "dev"
filepath = Path(DATA_DIR) / "annotated" / split / f"{filename}.html"

with open(filepath, "r", encoding="utf-8") as f:
    html_content = f.read()

### System prompt loading

In [ ]:
prompt_filename = "coref_long.txt"
with open(PROMPT_DIR / prompt_filename, "r", encoding="utf-8") as f:
    system_prompt = f.read()

print("system_prompt used : ", prompt_filename)



### Assistant loading

In [3]:
MODEL_MAPPING_NAME = {
    "qwen7b": "Qwen2.5-7B-Instruct",
    "qwen32b": "Qwen2.5-32B-Instruct",
    "gpt-5.2": "gpt-5.2",
    "phi-4": "phi-4",
    "saul-54b": "SaulLM-54B-Instruct"
}

from src.models import AssistantFactory

model = "gpt-5.2"

if model == "gpt-5.2":
        assistant = AssistantFactory.create_from_config({
            "type": "openai",
            "model_name": model,
            "temperature": 1,
        })
else:
    assistant = AssistantFactory.create(MODEL_MAPPING_NAME[model])

In [ ]:
class ReferenceProfile:
    def __init__(self, doctype, jurisdiction:str=None, main_title=None):
        self.doc_type = doctype
        self.jurisdiction = jurisdiction
        self.main_title = main_title
        self.alternative_titles = []
        self.citations = []
        self.fragments_mentioned = []
        self.authors = [] #Only for secondary sources

    def add_alternative_title(self, title):
        self.alternative_titles.append(title)
    
    def add_citation(self, citation):
        self.citations.append(citation)
    
    def add_fragment_mentioned(self, fragment):
        self.fragments_mentioned.append(fragment)
    
    def add_author(self, author):
        self.authors.append(author)

    def __str__(self):
        return f"ReferenceProfile(main_title={self.main_title}, doc_type={self.doc_type}, jurisdiction={self.jurisdiction}, alternative_titles={self.alternative_titles}, citations={self.citations}, fragments_mentioned={self.fragments_mentioned}, authors={self.authors})"


class ReferenceProfileList:
    def __init__(self):
        self.profiles = []
    
    def add_profile(self, profile: ReferenceProfile):
        self.profiles.append(profile)

    def update(self, parsed:str):
        # Parse the LLM output and update the reference profiles accordingly
        pass
    

### Chunking with the chunker

In [4]:
chunker = "sentence"  # "paragraph" | "sentence"

from src.chunkers.cache import cache_exists, load_cache
from src.chunkers import ChunkerFactory

if not cache_exists(chunker, split, filename):

    # Load spaCy only if needed
    nlp = None
    if chunker == "sentence":
        import spacy
        nlp = spacy.load("en_core_web_trf")
        print("✅ Model loaded.\n")


    token_chunks = ChunkerFactory.get_chunks(
        html_content, method=chunker, split=split, filename=filename, nlp=nlp
    )

else:
    token_chunks = load_cache(chunker, split, filename)




In [ ]:
def get_user_input_for_coref(input_token_chunk, list_mention, reference_profile_list):
    # Return a user string that includes the decoded token chunk, the list of references identified and the list of references mentioned in the previous chunks (from the reference profile list)
    return f"""Here is a chunk of text from a legal document: {decode(input_token_chunk)}.

References identified in this chunk:
{', '.join(list_mention)}

References mentioned in previous chunks:
{', '.join(str(profile) for profile in reference_profile_list.profiles)}"""

### Main processing function

In [ ]:
from src.models import get_messages
from tqdm import tqdm

reference_profile_list = ReferenceProfileList()
processed_chunks = []
for token_chunk in tqdm(token_chunks):

    list_mention_in_current_chunk = get_list_mention(token_chunk) # List of Mention object
    #This function already exists

    ############ Create the user message
    user_input =  get_user_input_for_coref(input_token_chunk=token_chunk, list_mention=list_mention_in_current_chunk,reference_profile_list=reference_profile_list)

    message = get_messages(system_prompt=system_prompt, user_input=user_input, fewshot_examples=final_fewshot, has_system_role=assistant.has_system_role)

    ############ Generate
    generated = assistant.generate(messages=message)

    parsed_generated = parse_coref_output(generated)
    # parsed_generated : {text_mention: main_title_group}

    # Little conversion {text_mention: main_title_group} -> {Mention object: main_title_group}
    converted_parsed_generated = {}
    for text_mention in parsed_generated.keys():
        mention_obj = next((mention for mention in list_mention_in_current_chunk if mention.text == text_mention), None) #might be too strick, we may consider a more flexible matching (e.g. partial match, or using character offsets if available)
        if mention_obj:
            converted_parsed_generated[mention_obj] = parsed_generated[text_mention]

    ############ Update the reference profile list
    reference_profile_list.update(converted_parsed_generated)

    token_chunk

    processed_chunks.append(corrected_generated_tokens)